# ML Assignment 2 — Dry Beans Classification
## Model Training Notebook

This notebook trains five classification models on the Dry Beans dataset (UCI ID: 602),
evaluates each with six metrics, and saves everything for the Streamlit app.

In [ ]:
import os, urllib.request, zipfile, warnings
import numpy as np, pandas as pd, joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, matthews_corrcoef)
warnings.filterwarnings('ignore')

MODEL_DIR = os.path.join(os.getcwd(), '..', 'model')
os.makedirs(MODEL_DIR, exist_ok=True)

## 1. Load Dataset

In [ ]:
zip_url = 'https://archive.ics.uci.edu/static/public/602/dry+bean+dataset.zip'
zip_path = 'dry_bean.zip'
if not os.path.exists(zip_path):
    urllib.request.urlretrieve(zip_url, zip_path)
if not os.path.exists('dry_bean_raw'):
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall('dry_bean_raw')
xlsx = [os.path.join(r,f) for r,_,fs in os.walk('dry_bean_raw') for f in fs if f.endswith('.xlsx')][0]
df = pd.read_excel(xlsx)
print(f'Shape: {df.shape}')
df.head()

## 2. Preprocess

In [ ]:
X = df.drop(columns=['Class'])
y = df['Class']
le = LabelEncoder()
y_enc = le.fit_transform(y)
joblib.dump(le, os.path.join(MODEL_DIR, 'label_encoder.joblib'))
joblib.dump(list(X.columns), os.path.join(MODEL_DIR, 'feature_names.joblib'))

X_tr, X_te, y_tr, y_te = train_test_split(X, y_enc, test_size=0.2, random_state=42, stratify=y_enc)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)
joblib.dump(scaler, os.path.join(MODEL_DIR, 'scaler.joblib'))
print(f'Train: {X_tr.shape}, Test: {X_te.shape}')
print(f'Classes: {le.classes_}')

## 3. Train Models & Evaluate

In [ ]:
models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42), True),
    'Decision Tree': (DecisionTreeClassifier(max_depth=10, random_state=42), False),
    'K-Nearest Neighbors': (KNeighborsClassifier(n_neighbors=7), True),
    'Naive Bayes (Gaussian)': (GaussianNB(), False),
    'Random Forest (Ensemble)': (RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1), False),
}

results = {}
for name, (model, scaled) in models.items():
    Xtr = X_tr_s if scaled else X_tr
    Xte = X_te_s if scaled else X_te
    model.fit(Xtr, y_tr)
    safe = name.lower().replace(' ','_').replace('(','').replace(')','')
    joblib.dump(model, os.path.join(MODEL_DIR, f'{safe}.joblib'))
    y_pred = model.predict(Xte)
    try:
        auc = roc_auc_score(y_te, model.predict_proba(Xte), multi_class='ovr', average='weighted')
    except: auc = float('nan')
    results[name] = {
        'Accuracy': accuracy_score(y_te, y_pred),
        'AUC': auc,
        'Precision': precision_score(y_te, y_pred, average='weighted'),
        'Recall': recall_score(y_te, y_pred, average='weighted'),
        'F1': f1_score(y_te, y_pred, average='weighted'),
        'MCC': matthews_corrcoef(y_te, y_pred),
    }
    print(f'{name}: {results[name]}')

In [ ]:
results_df = pd.DataFrame(results).T[['Accuracy','AUC','Precision','Recall','F1','MCC']]
results_df.to_csv(os.path.join(MODEL_DIR, 'metrics_summary.csv'))
results_df.style.format('{:.4f}')

## 4. Save Test Data

In [ ]:
test_df = pd.DataFrame(X_te, columns=list(X_tr.columns))
test_df['Class'] = y_te
test_df['Class_Name'] = le.inverse_transform(y_te)
test_df.to_csv(os.path.join(MODEL_DIR, '..', 'test_data.csv'), index=False)
print(f'Saved test_data.csv: {test_df.shape}')